# What Turns a Grocery Customer into a Regular?

**Notebook purpose:** Walk through the analysis step by step with plain-English explanations.
All heavy computation is done in `src/` scripts; this notebook loads the results and explains them.

**Data:** Instacart Online Grocery Shopping Dataset (Kaggle). 3.4M orders from 206K users.

> ⚠️ **Survivorship bias:** Every user in this data has ≥4 orders. Customers who tried the service once and left are NOT represented. All findings describe differences among already-retained customers.

In [ ]:
import json
import pandas as pd
import numpy as np
from IPython.display import Image, display, Markdown

# Load computed results
with open('../results/results.json') as f:
    R = json.load(f)

df = pd.read_parquet('../data/processed/user_features.parquet')
print(f"Loaded {len(df):,} users")

## 1. Data Reality Check

Before any analysis, we verified the data:

In [ ]:
p0 = R['phase0_data_reality_check']
print(f"Total orders: {p0['row_counts']['orders']:,}")
print(f"Total users:  {p0['n_users']:,}")
print(f"Products:     {p0['row_counts']['products']:,}")
print(f"Min orders/user: {p0['orders_per_user']['min']}")
print(f"Max orders/user: {p0['orders_per_user']['max']}")
print(f"\ndays_since_prior_order capped at 30: {p0['days_since_prior_order']['n_at_30']:,} orders at exactly 30")
print(f"  vs {p0['days_since_prior_order']['n_at_29']:,} at 29 ({p0['days_since_prior_order']['spike_ratio_30_vs_29']}x spike)")

## 2. Outcome Definitions

- **Heavy user:** Total orders ≥ 75th percentile (≥20 orders)
- **Fast cadence:** Median gap between orders 4+ ≤ 7 days

Features use orders 1–3 only. Outcomes use orders 4+ only. This prevents leakage.

In [ ]:
display(Image(filename='../results/figures/outcome_distributions.png', width=800))

## 3. Feature Comparisons: Heavy vs Non-Heavy

We compared early features between heavy and non-heavy users using bootstrap CIs and Cohen's d.
Effects are categorized using standard Cohen's d brackets: negligible (<0.2), small (0.2–0.5), medium (0.5–0.8), large (≥0.8).
Plain-English takeaway: all basket-composition and size effects fall in the small range (|d| < 0.20).

In [ ]:
display(Image(filename='../results/figures/heavy_vs_nonheavy_features.png', width=700))

**Key observation:** The gap features (days to 2nd/3rd order) have by far the largest effect sizes (d ≈ -0.7), but they partly restate the outcome — users who return quickly naturally accumulate more orders. Among non-gap features, first-order size, average basket size, morning-order share, aisle count, and produce/dairy shares show the largest (though still small) effects.

## 4. Logistic Regression

To eliminate multicollinearity (VIFs > 10 between first-order size, department count, and aisle count), we dropped `first_order_size` and `n_aisles_order1` from the logistic models. All remaining VIFs are strictly < 2.5, with zero sign flips.

We fit two logistic regressions:
1. **Without gap features** (lead model) — to evaluate early basket and time-of-day signals. Cross-validated AUC is ~0.57, reflecting that early basket features alone offer modest descriptive discrimination.
2. **With gap features** — adding early pacing jumps AUC to ~0.76, but this signal is partly circular (order count builds up with how often someone orders).

In [ ]:
display(Image(filename='../results/figures/odds_ratios_no_gap.png', width=700))
print("Lead model (without gap features)")

In [ ]:
display(Image(filename='../results/figures/odds_ratios_with_gap.png', width=700))
print("Secondary model (with gap features)")

## 5. Heavy × Fast-Cadence Overlap

In [ ]:
overlap = R['phase2_analysis']['heavy_fast_overlap']
print(f"Both heavy AND fast-cadence: {overlap['both']:,}")
print(f"Heavy only:                  {overlap['heavy_only']:,}")
print(f"Fast-cadence only:           {overlap['fast_only']:,}")
print(f"Neither:                     {overlap['neither']:,}")
print(f"\n{overlap['pct_overlap_among_heavy']}% of heavy users are also fast-cadence")
print(f"{overlap['pct_overlap_among_fast']}% of fast-cadence users are also heavy")

## 6. Sensitivity Checks

We re-ran the regression with top-20% and top-30% thresholds. The same features rank similarly across all three thresholds — the findings are not an artifact of one cutoff.

In [ ]:
display(Image(filename='../results/figures/sensitivity_check.png', width=800))

## 7. Lifecycle Segments

Defined from orders 1–3 only (basket size and early gap), so they can be applied at the moment a user completes their 3rd order.

In [ ]:
display(Image(filename='../results/figures/lifecycle_segments.png', width=800))

## 8. Reorder-Rate Curve

Plotted for orders 2 to 10 for a constant cohort of 101,696 users with ≥10 prior orders. The rise from 28.2% to 63.7% is partly mechanical, as each order draws from an expanding history of previously purchased items.

In [ ]:
display(Image(filename='../results/figures/reorder_rate_curve.png', width=600))

## 9. Limitations

1. **Survivorship bias**: Only users with ≥4 orders are present. We cannot study one-time dropouts.
2. **Total orders capped at 100**: Right-censored at 100 orders.
3. **Median gap capped at 30 days**: Long gaps are truncated at 30.
4. No calendar dates, prices, revenue, or user demographics.
5. All findings are **descriptive, not causal**.
6. Single retailer, single time period.

See `docs/INTERVIEW_NOTES.md` for detailed interview talking points.